In [1]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from uuid import uuid4

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# 12 documents: ML (highly relevant to query), economics (somewhat relevant), cooking and sports (off-topic)
# The threshold filter will progressively cut out the lower-scoring documents
docs = [
    Document(page_content="Supervised learning trains models on labeled input-output pairs to predict unseen data.", metadata={"topic": "ml"}),
    Document(page_content="Neural networks learn by adjusting weights through backpropagation and gradient descent.", metadata={"topic": "ml"}),
    Document(page_content="Overfitting occurs when a model memorises training data and performs poorly on new data.", metadata={"topic": "ml"}),
    Document(page_content="Training data quality and quantity are the most important factors in model performance.", metadata={"topic": "ml"}),
    Document(page_content="Cross-validation splits data into folds to evaluate model generalisation more reliably.", metadata={"topic": "ml"}),
    Document(page_content="Inflation is the rate at which the general level of prices for goods and services rises over time.", metadata={"topic": "economics"}),
    Document(page_content="GDP measures the total monetary value of all goods and services produced within a country.", metadata={"topic": "economics"}),
    Document(page_content="Interest rates set by central banks influence borrowing costs and consumer spending.", metadata={"topic": "economics"}),
    Document(page_content="Caramelisation occurs when sugar is heated above 160°C, creating complex flavour compounds.", metadata={"topic": "cooking"}),
    Document(page_content="Fermentation uses microorganisms to convert sugars into alcohol or acids, preserving food.", metadata={"topic": "cooking"}),
    Document(page_content="A marathon is a long-distance race of exactly 42.195 kilometres, run on roads.", metadata={"topic": "sports"}),
    Document(page_content="Tennis scoring follows a love, 15, 30, 40, game sequence, with deuce at 40-40.", metadata={"topic": "sports"}),
]

In [3]:
for ind, doc in enumerate(docs, 1):
    print(f"Doc No.- {ind} | MetaData- {doc.metadata}")
    print(f"Page Content: {doc.page_content}")
    print("-"*60)

Doc No.- 1 | MetaData- {'topic': 'ml'}
Page Content: Supervised learning trains models on labeled input-output pairs to predict unseen data.
------------------------------------------------------------
Doc No.- 2 | MetaData- {'topic': 'ml'}
Page Content: Neural networks learn by adjusting weights through backpropagation and gradient descent.
------------------------------------------------------------
Doc No.- 3 | MetaData- {'topic': 'ml'}
Page Content: Overfitting occurs when a model memorises training data and performs poorly on new data.
------------------------------------------------------------
Doc No.- 4 | MetaData- {'topic': 'ml'}
Page Content: Training data quality and quantity are the most important factors in model performance.
------------------------------------------------------------
Doc No.- 5 | MetaData- {'topic': 'ml'}
Page Content: Cross-validation splits data into folds to evaluate model generalisation more reliably.
-------------------------------------------------

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    embedding_function= embeddings,
    collection_name="demo",
    collection_configuration={
        'hnsw': {
            "space":"cosine" #l2, ip, cosine
        }
    }
)

In [5]:
ids_added = vectorstore.add_documents(
    documents=docs,
    ids=[str(uuid4()) for _ in range(len(docs))]
    )

len(ids_added)

12

In [6]:
len(docs)

12

In [21]:
query = "How ML model training works"

threshold = 0.4

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs= {'score_threshold': threshold}
)

results = retriever.invoke(query)
results

[Document(id='e9cab44e-7406-45db-91c6-8d0b446fe956', metadata={'topic': 'ml'}, page_content='Training data quality and quantity are the most important factors in model performance.'),
 Document(id='3ed0d121-5f23-4459-94f1-f86142114324', metadata={'topic': 'ml'}, page_content='Overfitting occurs when a model memorises training data and performs poorly on new data.'),
 Document(id='730575da-5cc4-4f80-8ecb-796b9bdee3b5', metadata={'topic': 'ml'}, page_content='Supervised learning trains models on labeled input-output pairs to predict unseen data.')]

In [22]:
similarity_score = vectorstore.similarity_search_with_score(query=query, k=4)

In [23]:
similarity_score

[(Document(id='e9cab44e-7406-45db-91c6-8d0b446fe956', metadata={'topic': 'ml'}, page_content='Training data quality and quantity are the most important factors in model performance.'),
  0.5605266094207764),
 (Document(id='3ed0d121-5f23-4459-94f1-f86142114324', metadata={'topic': 'ml'}, page_content='Overfitting occurs when a model memorises training data and performs poorly on new data.'),
  0.5862763524055481),
 (Document(id='730575da-5cc4-4f80-8ecb-796b9bdee3b5', metadata={'topic': 'ml'}, page_content='Supervised learning trains models on labeled input-output pairs to predict unseen data.'),
  0.5868589878082275),
 (Document(id='8b1ef73e-1d06-4960-ae13-1ca17120fc6e', metadata={'topic': 'ml'}, page_content='Neural networks learn by adjusting weights through backpropagation and gradient descent.'),
  0.6165425777435303)]

In [24]:
for doc, score in similarity_score:
    print(f"Score- {score} | Metadata- {doc.metadata}")
    print(doc.page_content)
    print("\n--\n")

Score- 0.5605266094207764 | Metadata- {'topic': 'ml'}
Training data quality and quantity are the most important factors in model performance.

--

Score- 0.5862763524055481 | Metadata- {'topic': 'ml'}
Overfitting occurs when a model memorises training data and performs poorly on new data.

--

Score- 0.5868589878082275 | Metadata- {'topic': 'ml'}
Supervised learning trains models on labeled input-output pairs to predict unseen data.

--

Score- 0.6165425777435303 | Metadata- {'topic': 'ml'}
Neural networks learn by adjusting weights through backpropagation and gradient descent.

--



In [31]:
# Cosine Score for each Document
for _, score in similarity_score:
    print(f"Cosine Score: {1-score:.4f}")

Cosine Score: 0.4395
Cosine Score: 0.4137
Cosine Score: 0.4131
Cosine Score: 0.3835


In [32]:
query = "How ML model training works"

threshold = 0.4

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs= {'score_threshold': threshold}
)

results = retriever.invoke(query)
results

[Document(id='e9cab44e-7406-45db-91c6-8d0b446fe956', metadata={'topic': 'ml'}, page_content='Training data quality and quantity are the most important factors in model performance.'),
 Document(id='3ed0d121-5f23-4459-94f1-f86142114324', metadata={'topic': 'ml'}, page_content='Overfitting occurs when a model memorises training data and performs poorly on new data.'),
 Document(id='730575da-5cc4-4f80-8ecb-796b9bdee3b5', metadata={'topic': 'ml'}, page_content='Supervised learning trains models on labeled input-output pairs to predict unseen data.')]